In [ ]:
%pip install scikit-survival

In [8]:
pip install scikit-survival xgboost

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 26.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 61.9 MB/s  0:00:00
  Created wheel for ecos: filename=ecos-2.0.14-cp313-cp313-linux_x86_64.whl size=204001 sha256=c0c3fc5a3b3ba6a63b7832d9b353f5bfe643d5ff7fbd50d31d8d6c67d8fc7969
  Stored in directory: /root/.cache/pip/wheels/6b/82/0b/4bb5aa6c4618f367601a45db8710205d56858808423974e92a
Successfully built ecos
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-survival]


In [2]:
from google.colab import files

uploaded = files.upload()

Saving actg320_clean.csv to actg320_clean.csv


In [3]:
import pandas as pd

df = pd.read_csv("actg320_clean.csv")

print(df.shape)
print(df.head())

(1151, 13)
   time  censor  treatment  treatment_group  strat2  sex  raceth  ivdrug  \
0   189       0          0                1       1    1       1       1   
1   287       0          0                1       1    2       2       1   
2   242       0          1                2       0    1       1       1   
3   199       0          0                1       1    1       1       1   
4   286       0          1                2       0    1       1       3   

   hemophil  karnof    cd4  priorzdv  age  
0         0     100  169.0      39.0   34  
1         0      90  149.5      15.0   34  
2         1     100   23.5       9.0   20  
3         0      90   46.0      53.0   48  
4         0      90   10.0      12.0   46  


In [4]:

categorical = [
    "treatment", "treatment_group", "strat2",
    "sex", "raceth", "ivdrug", "hemophil"
]

numerical = [
    "karnof", "cd4", "priorzdv", "age"
]

X = pd.get_dummies(
    df[categorical + numerical],
    columns=categorical,
    drop_first=True
)

y_time = df["time"].astype(float)
y_event = df["censor"].astype(bool)

In [5]:
print(X.dtypes)

karnof                 int64
cd4                  float64
priorzdv             float64
age                    int64
treatment_1             bool
treatment_group_2       bool
treatment_group_3       bool
treatment_group_4       bool
strat2_1                bool
sex_2                   bool
raceth_2                bool
raceth_3                bool
raceth_4                bool
raceth_5                bool
ivdrug_2                bool
ivdrug_3                bool
hemophil_1              bool
dtype: object


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1151 entries, 0 to 1150
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   time             1151 non-null   int64  
 1   censor           1151 non-null   int64  
 2   treatment        1151 non-null   int64  
 3   treatment_group  1151 non-null   int64  
 4   strat2           1151 non-null   int64  
 5   sex              1151 non-null   int64  
 6   raceth           1151 non-null   int64  
 7   ivdrug           1151 non-null   int64  
 8   hemophil         1151 non-null   int64  
 9   karnof           1151 non-null   int64  
 10  cd4              1151 non-null   float64
 11  priorzdv         1151 non-null   float64
 12  age              1151 non-null   int64  
dtypes: float64(2), int64(11)
memory usage: 117.0 KB


In [9]:
from sksurv.util import Surv

y = Surv.from_arrays(
    event=df["censor"].astype(bool),
    time=df["time"]
)

In [13]:
from sklearn.model_selection import train_test_split
X = pd.get_dummies(
    df.drop(columns=["time", "censor"]),
    columns=[
        "treatment", "treatment_group", "strat2",
        "sex", "raceth", "ivdrug", "hemophil"
    ],
    drop_first=True
)

X = X.astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=df["censor"]
)

In [14]:
from sksurv.ensemble import RandomSurvivalForest

rsf = RandomSurvivalForest(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rsf.fit(X_train, y_train)

RandomSurvivalForest(n_estimators=300, n_jobs=-1, random_state=42)

In [15]:
risk_rsf = rsf.predict(X_test)

In [16]:
from sksurv.metrics import concordance_index_censored

cindex_rsf = concordance_index_censored(
    y_test["event"],
    y_test["time"],
    risk_rsf
)[0]

print("RSF C-index:", cindex_rsf)

RSF C-index: 0.7830214266341199


In [25]:
xgb_model = xgb.XGBRegressor(
    objective="survival:cox",
    n_estimators=500,
    learning_rate=0.02,
    max_depth=2,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    random_state=42
)



import numpy as np

# Encode event and censoring information
y_train_xgb = np.where(
    y_train["event"] == True,
    y_train["time"],
    -y_train["time"]
)

# Train the model
xgb_model.fit(
    X_train,
    y_train_xgb
)
risk_xgb = xgb_model.predict(X_test)

cindex_xgb = concordance_index_censored(
    y_test["event"],
    y_test["time"],
    risk_xgb
)[0]

print("XGBoost C-index:", cindex_xgb)

XGBoost C-index: 0.8158394358557093
